In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    'name_first' : 'first_name',
    'name_last' : 'last_name',
    'dob' : 'birth_date',
    'ioc' : 'country'
}

#Reading from Bronze

In [0]:
df = spark.table('tennislakehouse.bronze.players')

#Cleaning the data

In [0]:
%skip
%sql
--query for showing fuzzy duplicates
SELECT
  *
FROM tennislakehouse.bronze.players
WHERE wikidata_id IN (
  SELECT 
    wikidata_id
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (PARTITION BY wikidata_id ORDER BY wikidata_id) flag
    FROM tennislakehouse.bronze.players
    WHERE wikidata_id IS NOT NULL
  )
  WHERE flag != 1
)
ORDER BY wikidata_id

In [0]:
#resolving the "fuzzy duplicates" which have the same wikidata_id but slightly different values
w = Window.partitionBy('wikidata_id').orderBy(
    F.length('name_first').desc_nulls_last(),
    F.length('name_last').desc_nulls_last(),
    F.col('dob').desc_nulls_last(),
    F.col('height').desc_nulls_last()
)

df_ranked = df.withColumn("rank", F.row_number().over(w))
df = df_ranked.filter(F.col("rank") == 1).drop("rank")

In [0]:
#cleaning name_first
df = df.withColumn(
    "name_first",
    F.when(
        F.col("name_first").rlike(r"^[A-Za-z]\s*[A-Za-z]$"),
        F.concat_ws(
            " ",
            F.upper(F.regexp_extract("name_first", r"^([A-Za-z])", 1)),
            F.upper(F.regexp_extract("name_first", r"[A-Za-z]\s*([A-Za-z])$", 1))
        )
    ).otherwise(F.trim(F.col("name_first")))
)
df.fillna({'name_first' : 'N/A'})

In [0]:
#cleaning name_last
df = df.withColumn(
    'name_last',
    F.trim(F.col('name_last'))
)
df.fillna({'name_last' : 'N/A'})

In [0]:
#cleaning hand
df = df.withColumn(
    "hand",
    F.when(
        (F.col("hand").isNull()) | (F.col("hand").isin("U")),
        "N/A"
    ).otherwise(F.trim(F.col("hand")))
)

In [0]:
#cleaning dob
df = df.withColumn(
    'dob',
    F.try_to_date(F.col("dob").cast('bigint').cast("string"), "yyyyMMdd")
)

In [0]:
#cleaning ioc
df = df.withColumn(
    'ioc',
    F.when(
        F.col('ioc').isin('?') | F.col('ioc').isNull(),
        'N/A'
    ).otherwise(F.trim(F.col('ioc')))
)

In [0]:
#cleaning height
df = df.withColumn(
  'height',
  F.when(
    F.col("height").isNull(),
    0
  ).otherwise(F.col('height'))
)

In [0]:
#cleaning wikidata_id
df = df.withColumn(
    'wikidata_id',
    F.when(
        F.col("wikidata_id").isNull(),
        'N/A'
    ).otherwise(F.trim(F.col("wikidata_id")))
)

#Renaming the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Loading into Silver table

In [0]:
(
df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable('tennislakehouse.silver.players')
)